In [217]:
import dimcli
import pandas as pd
import numpy as np
from pandas.io.json import json_normalize
from dimcli.shortcuts import dslquery, dslqueryall, chunks_of, normalize_key
dimcli.login()

DimCli v0.6.1 - Succesfully connected to <https://app.dimensions.ai> (method: dsl.ini file)


In [5]:
GRID_ID = "grid.7445.2"

In [7]:
grants = dslqueryall(f'search grants where research_orgs.id = "{GRID_ID}" return grants[id+investigator_details]')

1000 / 9848
2000 / 9848
3000 / 9848
4000 / 9848
5000 / 9848
6000 / 9848
7000 / 9848
8000 / 9848
9000 / 9848
9848 / 9848


In [221]:
grantsdf = grants.as_dataframe().set_index('id')
grantsdf['investigator_details'] = grantsdf['investigator_details'].replace(np.nan, False)
grantsdf

,investigator_details
id,
grant.8672070,"[{'role': 'PI', 'first_name': 'Daniel', 'middl..."
grant.8585235,False
grant.8587224,False
grant.8671895,"[{'role': 'PI', 'first_name': 'M.c.', 'middle_..."
grant.8584913,False
...,...
grant.5134390,"[{'last_name': 'Woollard', 'role': 'PI', 'midd..."
grant.6798374,"[{'last_name': 'Warboys', 'role': 'PI', 'middl..."
grant.8531121,"[{'last_name': 'Wojciak-Stothard', 'role': 'PI..."


In [232]:
def investigator_to_dict(grant):
    return {
        'investigator_id': grant['id'],
        'first_name': grant['first_name'],
        'last_name': grant['last_name'],
        'middle_name': grant['middle_name'],
        'role': grant['role'],
    }

def aff_to_dict(aff):
    return {
        'aff_id': aff['id'],
        'aff_name': aff['name'],
        'aff_state_code': aff['state_code'],
        'aff_state': aff['state'], 
        'aff_country_code': aff['country_code'],
        'aff_city_id': aff['city_id'],
        'aff_city': aff['city'], 
        'aff_country': aff['country'],
    }

def result():
    for grant in grantsdf.itertuples():
        if not grant.investigator_details:
            yield {'id': grant.Index}
        else:
            for investigator in grant.investigator_details:
                if not any([aff['id'] == GRID_ID for aff in (investigator['affiliations'] if 'affiliations' in investigator else [])]):
                    if 'affiliations' in investigator:
                        for aff in investigator['affiliations']:
                            yield {'id': grant.Index, **investigator_to_dict(investigator), **aff_to_dict(aff)}
                    else:
                        yield {'id': grant.Index, **investigator_to_dict(investigator), 'aff_id': np.nan}


res = pd.DataFrame(result()).set_index('id').drop_duplicates()
res.describe()

,investigator_id,first_name,last_name,middle_name,role,aff_id,aff_name,aff_state_code,aff_state,aff_country_code,aff_city_id,aff_city,aff_country
count,5620,5846,5846,4469,5846,2662,2769,57,2382,2715,2701,327,2443
unique,5108,2250,4133,467,2,330,502,23,24,38,207,124,44
top,ur.01022255644.17,David,Smith,,Co-PI,grid.83440.3b,University College London,US-CA,London,GB,2643743,Manchester,United Kingdom
freq,5,150,28,3057,3724,239,236,10,520,2268,549,77,2185


In [233]:
res.to_csv('grant_affiliations.csv')